# Autonomous Agentic SEO Blog Writer
**Real-Time US Trending Topics + 7 AI Agents + Beautiful Gradio UI**

Stack: LangGraph + Groq + Serper API + Gradio
AI Credits: Claude (Anthropic) - Copilot (Microsoft) - Gemini (Google) - ChatGPT (OpenAI) - Perplexity AI

## Step 1 - Install

In [1]:
!pip install -q groq langgraph langchain langchain-groq langchain-community gradio requests
print('Done!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
Done!


## Step 2 - Imports

In [3]:
import os, re, json, time, warnings, requests, getpass
from datetime import datetime
from typing import TypedDict, List, Dict, Optional
warnings.filterwarnings('ignore')
from groq import Groq
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END
import gradio as gr
print('Imports OK | Gradio', gr.__version__)

Imports OK | Gradio 5.50.0


## Step 3 - API Keys

In [ ]:
GROQ_API_KEY   = getpass.getpass('Groq API Key   -> https://console.groq.com : ')
SERPER_API_KEY = getpass.getpass('Serper API Key -> https://serper.dev        : ')
os.environ['GROQ_API_KEY']   = GROQ_API_KEY
os.environ['SERPER_API_KEY'] = SERPER_API_KEY
_g = Groq(api_key=GROQ_API_KEY)
_t = _g.chat.completions.create(model='llama-3.1-8b-instant',
    messages=[{'role':'user','content':'Say OK'}], max_tokens=3)
print('Groq  ->', _t.choices[0].message.content.strip())
_r = requests.post('https://google.serper.dev/search',
    headers={'X-API-KEY': SERPER_API_KEY, 'Content-Type': 'application/json'},
    json={'q':'test','num':1}, timeout=10)
print('Serper -> HTTP', _r.status_code)

## Step 4 - Helpers, State, All 7 Agents, LangGraph Pipeline

In [6]:
# ================================================================
# CONFIGURATION
# ================================================================
MODEL = 'llama-3.1-8b-instant'
TODAY = datetime.now().strftime('%B %d, %Y')

def get_llm(t=0.2, m=2048):
    return ChatGroq(model=MODEL, temperature=t, max_tokens=m, api_key=GROQ_API_KEY)

def llm_call(prompt, t=0.2, m=2048):
    chain = ChatPromptTemplate.from_template('{x}') | get_llm(t, m) | StrOutputParser()
    return chain.invoke({'x': prompt})

def web_search(q, n=6):
    try:
        r = requests.post('https://google.serper.dev/search',
            headers={'X-API-KEY': SERPER_API_KEY, 'Content-Type': 'application/json'},
            json={'q': q, 'num': n, 'gl': 'us', 'hl': 'en'}, timeout=15)
        if r.status_code != 200: return []
        return [{'title': x.get('title',''), 'link': x.get('link',''), 'snippet': x.get('snippet','')}
                for x in r.json().get('organic', [])[:n]]
    except: return []

def news_search(q, n=6):
    try:
        r = requests.post('https://google.serper.dev/news',
            headers={'X-API-KEY': SERPER_API_KEY, 'Content-Type': 'application/json'},
            json={'q': q, 'num': n, 'gl': 'us', 'hl': 'en'}, timeout=15)
        if r.status_code != 200: return []
        return [{'title': x.get('title',''), 'snippet': x.get('snippet',''),
                 'source': x.get('source',''), 'date': x.get('date','')}
                for x in r.json().get('news', [])[:n]]
    except: return []

def wlog(state, agent, msg):
    ts  = datetime.now().strftime('%H:%M:%S')
    log = list(state.get('agent_log', []))
    entry = '[' + ts + '] ' + agent.ljust(22) + '-> ' + msg
    log.append(entry)
    print('  ' + entry)
    return log

# ================================================================
# LANGGRAPH STATE
# ================================================================
class S(TypedDict):
    niche             : str
    trending_topics   : List[Dict]
    scored_topics     : List[Dict]
    selected_topic    : str
    topic_rationale   : str
    research_data     : str
    key_stats         : List[str]
    expert_quotes     : List[str]
    sources           : List[str]
    primary_keyword   : str
    secondary_keywords: List[str]
    longtail_keywords : List[str]
    meta_title        : str
    meta_description  : str
    url_slug          : str
    blog_outline      : str
    draft_post        : str
    final_post        : str
    seo_score         : int
    readability_score : int
    twitter_caption   : str
    linkedin_caption  : str
    instagram_caption : str
    agent_log         : List[str]
    status            : str
    start_time        : float

# ================================================================
# AGENT 1 - TREND SCOUT (REAL-TIME US NEWS TODAY)
# ================================================================
def trend_scout(state):
    log = wlog(state, 'TREND SCOUT', 'Fetching REAL-TIME US trends for ' + TODAY)
    niche = state.get('niche', 'technology AI business')

    # Real-time US trending queries - dated for today
    queries = [
        niche + ' breaking news United States today ' + TODAY,
        niche + ' trending US news this week March 2026',
        'top ' + niche + ' stories USA right now',
        niche + ' viral trending American news today',
        'Google trending ' + niche + ' United States 2026',
    ]

    all_web, all_news = [], []
    for q in queries:
        all_web.extend(web_search(q, 4))
        all_news.extend(news_search(q, 4))
        time.sleep(0.25)

    web_txt  = chr(10).join(['- ' + r['title'] + ': ' + r['snippet'][:100] for r in all_web[:18]])
    news_txt = chr(10).join(['- [' + n['date'] + '] ' + n['title'] + ' (' + n['source'] + '): ' + n['snippet'][:80] for n in all_news[:15]])

    prompt = (
        'You are a US trending content analyst. Today is ' + TODAY + '.\n'
        'Based on REAL current search results and news, identify 8 hot trending blog topics\n'
        'for US readers right now in niche: ' + niche + '\n\n'
        'Live search results:\n' + web_txt + '\n\n'
        'Breaking news:\n' + news_txt + '\n\n'
        'Return ONLY a JSON array of 8 objects, no markdown, no extra text:\n'
        '[{"topic":"specific trending topic title","angle":"unique content angle",'
        '"why_trending":"why trending in US today","urgency":"high/medium"}]'
    )
    raw = llm_call(prompt, 0.3)
    try:
        topics = json.loads(re.sub('```json|```', '', raw).strip())
        if not isinstance(topics, list): raise ValueError()
    except:
        topics = [
            {'topic':'Agentic AI Reshaping US Workforce 2026','angle':'Impact analysis','why_trending':'Explosive adoption','urgency':'high'},
            {'topic':'OpenAI GPT-5 Release Date and Features','angle':'What to expect','why_trending':'Most searched AI topic','urgency':'high'},
            {'topic':'US Tariff Impact on Tech Companies','angle':'Business strategy','why_trending':'Breaking economic news','urgency':'high'},
            {'topic':'Microsoft Copilot vs Google Gemini 2026','angle':'Head-to-head review','why_trending':'Enterprise AI battle','urgency':'medium'},
        ]

    log = wlog({'agent_log':log}, 'TREND SCOUT', 'Found ' + str(len(topics)) + ' real-time trending topics')
    return {**state, 'trending_topics': topics, 'agent_log': log, 'status': 'trends_found'}


# ================================================================
# AGENT 2 - TOPIC ANALYST
# ================================================================
def topic_analyst(state):
    log = wlog(state, 'TOPIC ANALYST', 'Scoring ' + str(len(state['trending_topics'])) + ' real-time topics')
    prompt = (
        'SEO strategist targeting US Google traffic. Today is ' + TODAY + '.\n'
        'Score each topic 0-10: search_volume, competition(lower=better), us_relevance, monetisation, timeliness.\n'
        'Topics: ' + json.dumps(state['trending_topics']) + '\n'
        'Return ONLY valid JSON, no markdown:\n'
        '{"scored":[{"topic":"...","total_score":8.5,"verdict":"..."}],'
        '"winner":"exact topic string","rationale":"2-sentence SEO reason"}'
    )
    raw = llm_call(prompt, 0.1)
    try:
        res = json.loads(re.sub('```json|```', '', raw).strip())
        scored    = res.get('scored', [])
        winner    = res.get('winner', state['trending_topics'][0]['topic'])
        rationale = res.get('rationale', 'Top trending US topic.')
    except:
        scored, winner, rationale = [], state['trending_topics'][0]['topic'], 'Top topic.'

    log = wlog({'agent_log':log}, 'TOPIC ANALYST', 'Winner: ' + winner[:55])
    return {**state, 'scored_topics': scored, 'selected_topic': winner,
            'topic_rationale': rationale, 'agent_log': log, 'status': 'topic_selected'}


# ================================================================
# AGENT 3 - RESEARCHER
# ================================================================
def researcher(state):
    topic = state['selected_topic']
    log   = wlog(state, 'RESEARCHER', 'Deep research: ' + topic[:50])
    queries = [
        topic + ' latest data statistics 2026',
        topic + ' expert analysis US market today',
        topic + ' impact American businesses',
        topic + ' trends forecast United States',
    ]
    snippets, srcs = [], []
    for q in queries:
        for r in web_search(q, 4):
            snippets.append('SOURCE: ' + r['title'] + chr(10) + r['snippet'])
            if r['link']: srcs.append(r['title'] + ' | ' + r['link'])
        time.sleep(0.25)

    raw_r = chr(10).join(snippets[:16])
    prompt = (
        'Research analyst. Extract structured data about: ' + topic + chr(10)
        + 'Sources: ' + raw_r[:3500] + chr(10)
        + 'Return ONLY valid JSON, no markdown: '
        + '{"summary":"3-paragraph detailed summary",'
        + '"key_stats":["stat with specific number","stat2","stat3","stat4","stat5"],'
        + '"expert_quotes":["expert insight 1","insight 2","insight 3"],'
        + '"key_findings":["finding1","finding2","finding3"],'
        + '"us_angle":"why this matters to American readers and businesses"}'
    )
    raw = llm_call(prompt, 0.1, 1500)
    try:
        d = json.loads(re.sub('```json|```', '', raw).strip())
        summary   = d.get('summary', '')
        key_stats = d.get('key_stats', [])
        quotes    = d.get('expert_quotes', [])
        findings  = d.get('key_findings', [])
        us_angle  = d.get('us_angle', '')
    except:
        summary, key_stats, quotes, findings, us_angle = raw_r[:600], [], [], [], ''

    rt = ('SUMMARY:' + chr(10) + summary + chr(10)*2
          + 'KEY STATS:' + chr(10) + chr(10).join(['- '+s for s in key_stats]) + chr(10)*2
          + 'US ANGLE: ' + us_angle)

    log = wlog({'agent_log':log}, 'RESEARCHER', str(len(key_stats)) + ' stats | ' + str(len(srcs)) + ' sources collected')
    return {**state, 'research_data': rt, 'key_stats': key_stats, 'expert_quotes': quotes,
            'sources': srcs[:8], 'agent_log': log, 'status': 'researched'}


# ================================================================
# AGENT 4 - SEO STRATEGY
# ================================================================
def seo_strategy(state):
    topic = state['selected_topic']
    log   = wlog(state, 'SEO STRATEGY', 'Building keywords + structure')
    comp  = web_search(topic + ' guide 2026 USA', 4)
    comp_t = chr(10).join(['- ' + r['title'] for r in comp])
    prompt = (
        'Senior SEO strategist, US Google traffic. Today: ' + TODAY + chr(10)
        + 'Topic: ' + topic + chr(10)
        + 'Research snippet: ' + state['research_data'][:400] + chr(10)
        + 'Competitor titles: ' + comp_t + chr(10)
        + 'Return ONLY valid JSON, no markdown: '
        + '{"primary_keyword":"main keyword phrase",'
        + '"secondary_keywords":["kw2","kw3","kw4","kw5"],'
        + '"longtail_keywords":["long tail 1 with US angle","long tail 2","long tail 3"],'
        + '"meta_title":"compelling title under 60 chars",'
        + '"meta_description":"persuasive desc under 155 chars",'
        + '"url_slug":"seo-url-slug",'
        + '"blog_outline":"H1: Title\\nH2: Hook Intro\\nH2: What Is It\\nH3: Key Aspects\\nH2: Why It Matters\\nH2: Key Statistics\\nH2: Real Examples\\nH2: How To Act Now\\nH2: Challenges\\nH2: Future Outlook\\nH2: Conclusion"}'
    )
    raw = llm_call(prompt, 0.2, 1200)
    try:
        seo = json.loads(re.sub('```json|```', '', raw).strip())
    except:
        slug = re.sub(r'[^a-z0-9-]','', topic.lower().replace(' ','-'))[:50]
        seo  = {
            'primary_keyword': topic.lower(),
            'secondary_keywords': [topic+' guide', topic+' 2026', 'US '+topic, 'American '+topic],
            'longtail_keywords': ['how '+topic+' affects US businesses','best '+topic+' strategies 2026','why '+topic+' matters USA'],
            'meta_title': topic[:52]+': 2026 Guide',
            'meta_description': 'Complete guide to '+topic+'. Data-backed insights for US professionals.',
            'url_slug': slug,
            'blog_outline': 'H1: '+topic+'\nH2: Introduction\nH2: What Is It\nH2: Key Stats\nH2: US Impact\nH2: Examples\nH2: Action Steps\nH2: Conclusion'
        }
    log = wlog({'agent_log':log}, 'SEO STRATEGY', 'Primary: ' + seo.get('primary_keyword','') + ' | /' + seo.get('url_slug',''))
    return {**state,
            'primary_keyword'   : seo.get('primary_keyword',''),
            'secondary_keywords': seo.get('secondary_keywords',[]),
            'longtail_keywords' : seo.get('longtail_keywords',[]),
            'meta_title'        : seo.get('meta_title',''),
            'meta_description'  : seo.get('meta_description',''),
            'url_slug'          : seo.get('url_slug',''),
            'blog_outline'      : seo.get('blog_outline',''),
            'agent_log': log, 'status': 'seo_ready'}


# ================================================================
# AGENT 5 - BLOG WRITER
# ================================================================
def blog_writer(state):
    topic = state['selected_topic']
    log   = wlog(state, 'BLOG WRITER', 'Writing full post: ' + topic[:45])
    sec   = ', '.join(state.get('secondary_keywords',[]))
    lng   = ', '.join(state.get('longtail_keywords',[]))
    stats = chr(10).join(['- '+s for s in state.get('key_stats',[])[:5]])
    qts   = chr(10).join(['- '+q for q in state.get('expert_quotes',[])[:3]])

    prompt = (
        'Expert US SEO blog writer. Today is ' + TODAY + '.\n'
        'Write a COMPLETE, high-quality, publication-ready blog post.\n'
        'TOPIC: ' + topic + '\n'
        'PRIMARY KEYWORD: ' + state['primary_keyword'] + '\n'
        'SECONDARY KEYWORDS: ' + sec + '\n'
        'LONG-TAIL KEYWORDS: ' + lng + '\n'
        'OUTLINE:\n' + state['blog_outline'] + '\n'
        'RESEARCH:\n' + state['research_data'][:1800] + '\n'
        'STATS:\n' + stats + '\n'
        'INSIGHTS:\n' + qts + '\n'
        'WRITING RULES:\n'
        '1. Open with a shocking real stat or bold question hook\n'
        '2. Minimum 1500 words - write EVERY section in FULL\n'
        '3. Use ## H2 and ### H3 headings\n'
        '4. Use bullet points and numbered lists\n'
        '5. US focus: American companies, USD amounts, US statistics\n'
        '6. Tone: confident, clear, authoritative but approachable\n'
        '7. Reference todays date ' + TODAY + ' to show freshness\n'
        '8. Strong CTA at the end\n'
        '9. Include 5-item FAQ at the end\n'
        '10. Full markdown formatting\n'
        'Write the ENTIRE post now, do not stop early.'
    )
    draft = llm_call(prompt, 0.4, 2048)
    wc    = len(draft.split())
    log   = wlog({'agent_log':log}, 'BLOG WRITER', 'Draft complete: ' + str(wc) + ' words')
    return {**state, 'draft_post': draft, 'agent_log': log, 'status': 'draft_written'}


# ================================================================
# AGENT 6 - EDITOR
# ================================================================
def editor(state):
    log = wlog(state, 'EDITOR', 'Polishing, scoring, optimising')
    prompt = (
        'Professional editor + SEO auditor. Polish and score this post.\n'
        'PRIMARY KEYWORD: ' + state['primary_keyword'] + '\n'
        'DRAFT:\n' + state['draft_post'][:3200] + '\n'
        'Fix grammar, improve flow, ensure keyword density, verify FAQ exists.\n'
        'Return ONLY valid JSON, no markdown:\n'
        '{"edited_post":"complete polished post here",'
        '"seo_score":85,"readability_score":82,'
        '"editor_notes":"what was improved"}'
    )
    raw = llm_call(prompt, 0.1, 2048)
    try:
        res   = json.loads(re.sub('```json|```', '', raw).strip())
        final = res.get('edited_post', state['draft_post'])
        seo   = min(100, max(0, int(res.get('seo_score', 78))))
        read  = min(100, max(0, int(res.get('readability_score', 75))))
        notes = res.get('editor_notes', '')
    except:
        final, seo, read, notes = state['draft_post'], 78, 75, 'Refined'

    log = wlog({'agent_log':log}, 'EDITOR', 'SEO: ' + str(seo) + '/100  Readability: ' + str(read) + '/100')
    return {**state, 'final_post': final, 'seo_score': seo,
            'readability_score': read, 'agent_log': log, 'status': 'edited'}


# ================================================================
# AGENT 7 - SOCIAL MEDIA
# ================================================================
def social_media(state):
    topic = state['selected_topic']
    log   = wlog(state, 'SOCIAL MEDIA', 'Generating 3-platform captions')
    prompt = (
        'US social media expert. Date: ' + TODAY + '.\n'
        'Write captions for topic: ' + topic + '\n'
        'Keyword: ' + state['primary_keyword'] + '\n'
        'Blog intro: ' + state['final_post'][:350] + '\n'
        'Return ONLY valid JSON, no markdown:\n'
        '{"twitter":"under 280 chars, 2-3 hashtags, emoji, link CTA",'
        '"linkedin":"professional, 3 short paragraphs, data point, CTA, under 1200 chars",'
        '"instagram":"engaging, line breaks, 5 relevant hashtags, emoji"}'
    )
    raw = llm_call(prompt, 0.5, 800)
    try:
        c  = json.loads(re.sub('```json|```', '', raw).strip())
        tw = c.get('twitter',   topic + ' #AI #Tech #Business')
        li = c.get('linkedin',  'New post: ' + topic)
        ig = c.get('instagram', topic + chr(10) + '#AI #Tech #Business #USA #2026')
    except:
        tw = 'Breaking: ' + topic + ' #AI #Tech #2026'
        li = 'Just published: ' + topic
        ig = topic + chr(10) + '#AI #Technology #Business #USA #2026'

    elapsed = round(time.time() - state.get('start_time', time.time()), 1)
    log = wlog({'agent_log':log}, 'SOCIAL MEDIA', 'All 3 captions ready')
    log = wlog({'agent_log':log}, 'PIPELINE', 'COMPLETE in ' + str(elapsed) + 's')
    return {**state, 'twitter_caption': tw, 'linkedin_caption': li,
            'instagram_caption': ig, 'agent_log': log, 'status': 'complete'}


# ================================================================
# BUILD LANGGRAPH
# ================================================================
wf = StateGraph(S)
wf.add_node('trend_scout',   trend_scout)
wf.add_node('topic_analyst', topic_analyst)
wf.add_node('researcher',    researcher)
wf.add_node('seo_strategy',  seo_strategy)
wf.add_node('blog_writer',   blog_writer)
wf.add_node('editor',        editor)
wf.add_node('social_media',  social_media)
wf.set_entry_point('trend_scout')
wf.add_edge('trend_scout',   'topic_analyst')
wf.add_edge('topic_analyst', 'researcher')
wf.add_edge('researcher',    'seo_strategy')
wf.add_edge('seo_strategy',  'blog_writer')
wf.add_edge('blog_writer',   'editor')
wf.add_edge('editor',        'social_media')
wf.add_edge('social_media',  END)
pipeline = wf.compile()

print('Pipeline ready! Agents: Trend Scout -> Topic Analyst -> Researcher -> SEO -> Writer -> Editor -> Social')

Pipeline ready! Agents: Trend Scout -> Topic Analyst -> Researcher -> SEO -> Writer -> Editor -> Social


## Step 5 - Launch Dashboard

In [7]:
# ================================================================
# GRADIO BACKEND
# ================================================================
def run_pipeline(niche, extra, model_choice):
    if not niche.strip():
        return ('Please enter a niche.',)*7
    global MODEL
    MODEL = {'llama-3.1-8b-instant (Fast)':'llama-3.1-8b-instant',
             'llama-3.3-70b-versatile (Best)':'llama-3.3-70b-versatile',
             'mixtral-8x7b-32768 (Reasoning)':'mixtral-8x7b-32768'}.get(model_choice,'llama-3.1-8b-instant')
    try:
        s = pipeline.invoke({
            'niche'      : niche + ('. Focus: ' + extra if extra.strip() else ''),
            'agent_log'  : [],
            'status'     : 'init',
            'start_time' : time.time()
        })
        nl = chr(10)
        seo_card = (
            'PRIMARY KEYWORD' + nl + '  ' + s.get('primary_keyword','') + nl*2
            + 'SECONDARY KEYWORDS' + nl + '  ' + ', '.join(s.get('secondary_keywords',[])) + nl*2
            + 'LONG-TAIL KEYWORDS' + nl
            + nl.join(['  - '+k for k in s.get('longtail_keywords',[])]) + nl*2
            + 'META TITLE' + nl + '  ' + s.get('meta_title','') + nl*2
            + 'META DESCRIPTION' + nl + '  ' + s.get('meta_description','') + nl*2
            + 'URL SLUG' + nl + '  /' + s.get('url_slug','') + nl*2
            + 'SEO SCORE     : ' + str(s.get('seo_score',0)) + '/100' + nl
            + 'READABILITY   : ' + str(s.get('readability_score',0)) + '/100' + nl
            + 'WORD COUNT    : ' + str(len(s.get('final_post','').split()))
        )
        topics = sorted(s.get('scored_topics',[]), key=lambda x: x.get('total_score',0), reverse=True)
        research_card = (
            'SELECTED TODAY (' + TODAY + ')' + nl
            + '  ' + s.get('selected_topic','') + nl*2
            + 'RATIONALE' + nl + '  ' + s.get('topic_rationale','') + nl*2
            + 'ALL SCORED TOPICS' + nl
            + nl.join(['  ['+str(i+1)+'] '+str(t.get('total_score',0))+' pts | '+t.get('topic','')[:65] for i,t in enumerate(topics[:8])]) + nl*2
            + 'KEY STATS' + nl + nl.join(['  - '+x for x in s.get('key_stats',[])]) + nl*2
            + 'SOURCES' + nl + nl.join(['  - '+x[:80] for x in s.get('sources',[])])
        )
        social_card = (
            'TWITTER / X' + nl + s.get('twitter_caption','') + nl*2
            + '--- LINKEDIN ---' + nl + s.get('linkedin_caption','') + nl*2
            + '--- INSTAGRAM ---' + nl + s.get('instagram_caption','')
        )
        agent_log  = nl.join(s.get('agent_log',[]))
        blog_post  = s.get('final_post','No post generated.')
        outline    = s.get('blog_outline','')
        status_bar = 'DONE | Topic: ' + s.get('selected_topic','')[:60] + ' | SEO: ' + str(s.get('seo_score',0)) + '/100 | Words: ' + str(len(blog_post.split()))
        return blog_post, seo_card, research_card, social_card, agent_log, outline, status_bar
    except Exception as e:
        err = 'Error: ' + str(e)
        return (err,)*7

def save_md(text):
    if not text or text.startswith('Error') or len(text) < 100: return None
    fn = '/content/blog_' + str(int(time.time())) + '.md'
    open(fn,'w').write(text)
    return fn

NICHES = [
    'Technology & AI',
    'Finance & Economy',
    'Startups & Venture Capital',
    'Artificial Intelligence & Machine Learning',
    'Cryptocurrency & Blockchain',
    'SaaS & Productivity Tools',
    'Data Analytics & Business Intelligence',
    'Electric Vehicles & Clean Energy',
    'Health Tech & Digital Wellness',
    'E-Commerce & Digital Marketing',
    'Cybersecurity & Privacy',
    'Real Estate & PropTech',
]

# ================================================================
# CSS - DARK PROFESSIONAL THEME WITH HIGH CONTRAST
# ================================================================
CSS = (
    "@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800;900&family=JetBrains+Mono:wght@400;500;600&display=swap');"
    ":root{"
    "--bg:#0F1117;--surface:#1A1D27;--surface2:#22263A;--surface3:#2A2E42;"
    "--border:#2E3350;--border2:#3D4266;"
    "--blue:#3B82F6;--blue2:#60A5FA;--violet:#8B5CF6;--violet2:#A78BFA;"
    "--green:#10B981;--green2:#34D399;--amber:#F59E0B;--amber2:#FCD34D;"
    "--red:#EF4444;--pink:#EC4899;"
    "--text:#F1F5F9;--text2:#CBD5E1;--text3:#94A3B8;--text4:#64748B;"
    "}"
    "*{font-family:'Inter',sans-serif !important;}"
    "code,pre,.mono{font-family:'JetBrains Mono',monospace !important;}"
    "body,.gradio-container{"
    "background:var(--bg) !important;"
    "color:var(--text) !important;"
    "min-height:100vh;"
    "}"

    # All input/textbox areas - dark bg, light text
    "textarea,input[type='text'],input[type='search']{"
    "background:var(--surface2) !important;"
    "color:var(--text) !important;"
    "border:1.5px solid var(--border) !important;"
    "border-radius:8px !important;"
    "font-size:13px !important;"
    "line-height:1.7 !important;"
    "}"
    "textarea:focus,input:focus{"
    "border-color:var(--blue) !important;"
    "box-shadow:0 0 0 3px rgba(59,130,246,0.15) !important;"
    "}"
    "textarea::placeholder,input::placeholder{color:var(--text4) !important;}"

    # Labels
    ".label-wrap span,label span{"
    "color:var(--text2) !important;"
    "font-size:11px !important;"
    "font-weight:600 !important;"
    "text-transform:uppercase !important;"
    "letter-spacing:0.8px !important;"
    "}"

    # Dropdowns
    ".wrap,.wrap-inner,.secondary-wrap{"
    "background:var(--surface2) !important;"
    "color:var(--text) !important;"
    "border-color:var(--border) !important;"
    "}"
    "select option{background:var(--surface2) !important;color:var(--text) !important;}"

    # Buttons
    ".btn-launch{"
    "background:linear-gradient(135deg,#3B82F6 0%,#8B5CF6 100%) !important;"
    "color:white !important;border:none !important;"
    "border-radius:10px !important;font-weight:700 !important;"
    "font-size:14px !important;width:100% !important;"
    "padding:14px !important;"
    "box-shadow:0 4px 20px rgba(59,130,246,0.35) !important;"
    "transition:all 0.2s !important;"
    "}"
    ".btn-launch:hover{transform:translateY(-2px) !important;box-shadow:0 8px 28px rgba(59,130,246,0.5) !important;}"
    ".btn-ghost{"
    "background:var(--surface3) !important;"
    "color:var(--text2) !important;"
    "border:1px solid var(--border2) !important;"
    "border-radius:8px !important;font-weight:600 !important;"
    "}"
    ".btn-ghost:hover{background:var(--border) !important;}"

    # Blog output - special readable styling
    ".blog-box textarea{"
    "background:#111827 !important;"
    "color:#E2E8F0 !important;"
    "border:1.5px solid #1E40AF !important;"
    "font-size:14px !important;"
    "line-height:1.85 !important;"
    "font-family:'Inter',sans-serif !important;"
    "}"

    # Agent log - terminal style
    ".log-box textarea{"
    "background:#050A14 !important;"
    "color:#38BDF8 !important;"
    "border:1.5px solid #0C4A6E !important;"
    "font-family:'JetBrains Mono',monospace !important;"
    "font-size:12px !important;"
    "line-height:1.6 !important;"
    "}"

    # SEO box
    ".seo-box textarea{"
    "background:#0A1628 !important;"
    "color:#86EFAC !important;"
    "border:1.5px solid #14532D !important;"
    "font-family:'JetBrains Mono',monospace !important;"
    "font-size:12px !important;"
    "line-height:1.65 !important;"
    "}"

    # Research box
    ".research-box textarea{"
    "background:#0C0A1E !important;"
    "color:#C4B5FD !important;"
    "border:1.5px solid #4C1D95 !important;"
    "font-family:'JetBrains Mono',monospace !important;"
    "font-size:12px !important;"
    "line-height:1.65 !important;"
    "}"

    # Social box
    ".social-box textarea{"
    "background:#130A00 !important;"
    "color:#FCD34D !important;"
    "border:1.5px solid #78350F !important;"
    "font-size:13px !important;"
    "line-height:1.7 !important;"
    "}"

    # Outline box
    ".outline-box textarea{"
    "background:#0A1628 !important;"
    "color:#93C5FD !important;"
    "border:1.5px solid #1E3A5F !important;"
    "font-size:13px !important;"
    "line-height:1.7 !important;"
    "}"

    # Status bar
    ".status-box textarea{"
    "background:#052E16 !important;"
    "color:#4ADE80 !important;"
    "border:1.5px solid #14532D !important;"
    "font-family:'JetBrains Mono',monospace !important;"
    "font-size:12px !important;"
    "}"

    # Tabs
    ".tab-nav{background:var(--surface) !important;border-bottom:1px solid var(--border) !important;}"
    ".tab-nav button{color:var(--text3) !important;font-weight:600 !important;font-size:13px !important;border-radius:0 !important;}"
    ".tab-nav button.selected{color:var(--blue2) !important;border-bottom:2px solid var(--blue) !important;background:transparent !important;}"
    ".tab-nav button:hover{color:var(--text) !important;}"

    # Accordion / panels
    ".block,.panel,.gr-form{"
    "background:var(--surface) !important;"
    "border:1px solid var(--border) !important;"
    "border-radius:10px !important;"
    "}"

    # Slider
    "input[type=range]{accent-color:var(--blue) !important;}"
    ".svelte-1cl284r{color:var(--text2) !important;}"
)


# ================================================================
# GRADIO UI
# ================================================================
with gr.Blocks(css=CSS, title='Agentic SEO Blog Writer', theme=gr.themes.Base(
    primary_hue='blue', secondary_hue='violet',
    neutral_hue='slate',
    font=['Inter', gr.themes.GoogleFont('Inter'), 'sans-serif']
).set(
    body_background_fill='#0F1117',
    block_background_fill='#1A1D27',
    block_border_color='#2E3350',
    block_label_text_color='#CBD5E1',
    input_background_fill='#22263A',
    input_border_color='#2E3350',
    button_primary_background_fill='linear-gradient(135deg,#3B82F6,#8B5CF6)',
    button_primary_text_color='white',
)) as app:

    # ── HEADER ──────────────────────────────────────────────
    gr.HTML(
        '<div style="background:linear-gradient(135deg,#1E1B4B 0%,#1E3A8A 40%,#312E81 100%);'
        'border-radius:16px;padding:32px 40px;margin-bottom:8px;'
        'border:1px solid #3730A3;box-shadow:0 8px 32px rgba(59,130,246,0.2);">'
        '<div style="display:flex;align-items:center;gap:16px;margin-bottom:14px">'
        '<div style="font-size:44px;line-height:1">&#129302;</div>'
        '<div>'
        '<h1 style="margin:0;font-size:26px;font-weight:800;color:#F1F5F9;letter-spacing:-0.5px">'
        'Autonomous Agentic SEO Blog Writer</h1>'
        '<p style="margin:4px 0 0;color:#93C5FD;font-size:13px;font-weight:500">'
        'Real-Time US Trending Topics &#8226; 7 AI Agents &#8226; LangGraph &#8226; Groq &#8226; Serper Live Search'
        '</p></div></div>'
        '<div style="display:flex;gap:6px;flex-wrap:wrap;margin-bottom:12px">'
        '<span style="background:rgba(59,130,246,0.2);color:#93C5FD;padding:3px 10px;border-radius:20px;font-size:11px;font-weight:700;border:1px solid #1D4ED8">7 AI Agents</span>'
        '<span style="background:rgba(139,92,246,0.2);color:#C4B5FD;padding:3px 10px;border-radius:20px;font-size:11px;font-weight:700;border:1px solid #5B21B6">LangGraph</span>'
        '<span style="background:rgba(16,185,129,0.2);color:#6EE7B7;padding:3px 10px;border-radius:20px;font-size:11px;font-weight:700;border:1px solid #065F46">Live US Trends</span>'
        '<span style="background:rgba(245,158,11,0.2);color:#FCD34D;padding:3px 10px;border-radius:20px;font-size:11px;font-weight:700;border:1px solid #78350F">Real-Time Research</span>'
        '<span style="background:rgba(236,72,153,0.2);color:#F9A8D4;padding:3px 10px;border-radius:20px;font-size:11px;font-weight:700;border:1px solid #831843">1500+ Words</span>'
        '<span style="background:rgba(59,130,246,0.2);color:#93C5FD;padding:3px 10px;border-radius:20px;font-size:11px;font-weight:700;border:1px solid #1D4ED8">SEO Optimised</span>'
        '</div>'
        '<div style="display:flex;align-items:center;gap:12px">'
        '<span style="color:#475569;font-size:11px;font-weight:600">FLOW:</span>'
        '<span style="color:#60A5FA;font-size:11px">Trend Scout</span>'
        '<span style="color:#475569">&#8594;</span>'
        '<span style="color:#A78BFA;font-size:11px">Topic Analyst</span>'
        '<span style="color:#475569">&#8594;</span>'
        '<span style="color:#34D399;font-size:11px">Researcher</span>'
        '<span style="color:#475569">&#8594;</span>'
        '<span style="color:#60A5FA;font-size:11px">SEO Strategy</span>'
        '<span style="color:#475569">&#8594;</span>'
        '<span style="color:#F9A8D4;font-size:11px">Blog Writer</span>'
        '<span style="color:#475569">&#8594;</span>'
        '<span style="color:#FCD34D;font-size:11px">Editor</span>'
        '<span style="color:#475569">&#8594;</span>'
        '<span style="color:#F87171;font-size:11px">Social Media</span>'
        '</div>'
        '</div>'
    )

    with gr.Tabs() as tabs:

        # ── TAB 1: CONTROL PANEL ─────────────────────────────
        with gr.TabItem('&#9881; Control Panel'):
            with gr.Row(equal_height=False):

                # LEFT COLUMN
                with gr.Column(scale=1, min_width=300):
                    gr.HTML('<div style="background:#1A1D27;border:1px solid #2E3350;border-radius:10px;padding:14px 18px;margin-bottom:8px"><p style="color:#60A5FA;font-size:12px;font-weight:700;margin:0 0 4px;text-transform:uppercase;letter-spacing:1px">LIVE DATE</p><p style="color:#F1F5F9;font-size:18px;font-weight:800;margin:0">' + datetime.now().strftime('%B %d, %Y') + '</p><p style="color:#64748B;font-size:11px;margin:4px 0 0">Topics pulled from real-time US search</p></div>')

                    niche_dd = gr.Dropdown(
                        choices=NICHES,
                        value='Technology & AI',
                        label='Content Niche',
                        allow_custom_value=True,
                        info='Select or type a custom niche'
                    )
                    extra_in = gr.Textbox(
                        label='Extra Focus (optional)',
                        placeholder='e.g. target small business owners, include pricing data...',
                        lines=2
                    )
                    model_dd = gr.Dropdown(
                        choices=[
                            'llama-3.1-8b-instant (Fast)',
                            'llama-3.3-70b-versatile (Best)',
                            'mixtral-8x7b-32768 (Reasoning)',
                        ],
                        value='llama-3.1-8b-instant (Fast)',
                        label='Groq LLM Model'
                    )

                    gr.HTML('<div style="border-top:1px solid #2E3350;margin:14px 0"></div>')

                    gr.HTML(
                        '<div style="background:#0A1628;border:1px solid #1E3A5F;border-radius:10px;padding:14px 18px">'
                        '<p style="color:#60A5FA;font-size:11px;font-weight:700;margin:0 0 10px;text-transform:uppercase;letter-spacing:1px">7-AGENT PIPELINE</p>'
                        '<div style="display:flex;flex-direction:column;gap:6px">'
                        '<div style="display:flex;align-items:center;gap:8px">'
                        '<span style="background:#1D4ED8;color:#93C5FD;padding:2px 7px;border-radius:4px;font-size:10px;font-weight:700">1</span>'
                        '<span style="color:#CBD5E1;font-size:12px">Trend Scout <span style="color:#10B981">(Live Serper search)</span></span></div>'
                        '<div style="display:flex;align-items:center;gap:8px">'
                        '<span style="background:#5B21B6;color:#C4B5FD;padding:2px 7px;border-radius:4px;font-size:10px;font-weight:700">2</span>'
                        '<span style="color:#CBD5E1;font-size:12px">Topic Analyst <span style="color:#A78BFA">(5-dim scoring)</span></span></div>'
                        '<div style="display:flex;align-items:center;gap:8px">'
                        '<span style="background:#065F46;color:#6EE7B7;padding:2px 7px;border-radius:4px;font-size:10px;font-weight:700">3</span>'
                        '<span style="color:#CBD5E1;font-size:12px">Researcher <span style="color:#34D399">(4 search queries)</span></span></div>'
                        '<div style="display:flex;align-items:center;gap:8px">'
                        '<span style="background:#1D4ED8;color:#93C5FD;padding:2px 7px;border-radius:4px;font-size:10px;font-weight:700">4</span>'
                        '<span style="color:#CBD5E1;font-size:12px">SEO Strategy <span style="color:#60A5FA">(keywords + meta)</span></span></div>'
                        '<div style="display:flex;align-items:center;gap:8px">'
                        '<span style="background:#831843;color:#F9A8D4;padding:2px 7px;border-radius:4px;font-size:10px;font-weight:700">5</span>'
                        '<span style="color:#CBD5E1;font-size:12px">Blog Writer <span style="color:#F9A8D4">(1500+ words)</span></span></div>'
                        '<div style="display:flex;align-items:center;gap:8px">'
                        '<span style="background:#78350F;color:#FCD34D;padding:2px 7px;border-radius:4px;font-size:10px;font-weight:700">6</span>'
                        '<span style="color:#CBD5E1;font-size:12px">Editor <span style="color:#FCD34D">(SEO score)</span></span></div>'
                        '<div style="display:flex;align-items:center;gap:8px">'
                        '<span style="background:#7F1D1D;color:#FCA5A5;padding:2px 7px;border-radius:4px;font-size:10px;font-weight:700">7</span>'
                        '<span style="color:#CBD5E1;font-size:12px">Social Media <span style="color:#F87171">(3 platforms)</span></span></div>'
                        '</div></div>'
                    )

                    gr.HTML('<div style="border-top:1px solid #2E3350;margin:14px 0"></div>')
                    run_btn = gr.Button('LAUNCH AUTONOMOUS PIPELINE', elem_classes='btn-launch')
                    clr_btn = gr.Button('Clear All Outputs', elem_classes='btn-ghost')

                # RIGHT COLUMN - BLOG OUTPUT
                with gr.Column(scale=2):
                    status_out = gr.Textbox(
                        label='Pipeline Status',
                        lines=1, interactive=False,
                        placeholder='Status appears here when pipeline completes...',
                        elem_classes='status-box'
                    )
                    blog_out = gr.Textbox(
                        label='GENERATED BLOG POST  (SEO-Optimised | 1500+ Words | US-Focused)',
                        lines=32, interactive=False,
                        placeholder='Your real-time trending blog post will appear here...\n\nThe pipeline will:\n1. Scan live US news and search trends\n2. Select the best ranking opportunity\n3. Research with real data\n4. Build full SEO strategy\n5. Write 1500+ word post\n6. Polish and score it\n7. Generate social captions',
                        elem_classes='blog-box'
                    )
                    with gr.Row():
                        save_btn  = gr.Button('Save as Markdown (.md)', elem_classes='btn-ghost')
                        save_file = gr.File(label='Download', interactive=False)

        # ── TAB 2: SEO REPORT ────────────────────────────────
        with gr.TabItem('&#128200; SEO Report'):
            gr.HTML('<p style="color:#64748B;font-size:13px;padding:8px 4px">Complete SEO strategy generated by the SEO Strategy Agent.</p>')
            seo_out = gr.Textbox(
                label='SEO STRATEGY REPORT',
                lines=24, interactive=False,
                placeholder='Keywords, meta data, SEO scores appear here...',
                elem_classes='seo-box'
            )

        # ── TAB 3: TOPIC RESEARCH ────────────────────────────
        with gr.TabItem('&#128269; Topic Research'):
            gr.HTML('<p style="color:#64748B;font-size:13px;padding:8px 4px">Real-time trending topics discovered and scored by the Trend Scout and Topic Analyst agents.</p>')
            research_out = gr.Textbox(
                label='TOPIC SELECTION + RESEARCH DATA',
                lines=24, interactive=False,
                placeholder='Trending topics, scores, stats, and sources appear here...',
                elem_classes='research-box'
            )

        # ── TAB 4: SOCIAL MEDIA ──────────────────────────────
        with gr.TabItem('&#128241; Social Media'):
            gr.HTML('<p style="color:#64748B;font-size:13px;padding:8px 4px">Platform-specific captions generated by the Social Media Agent.</p>')
            social_out = gr.Textbox(
                label='SOCIAL MEDIA CAPTIONS  (Twitter / LinkedIn / Instagram)',
                lines=20, interactive=False,
                placeholder='Captions for all 3 platforms appear here...',
                elem_classes='social-box'
            )

        # ── TAB 5: OUTLINE ───────────────────────────────────
        with gr.TabItem('&#128221; Blog Outline'):
            gr.HTML('<p style="color:#64748B;font-size:13px;padding:8px 4px">Blog structure and H1/H2/H3 outline generated by the SEO Strategy Agent.</p>')
            outline_out = gr.Textbox(
                label='BLOG OUTLINE',
                lines=20, interactive=False,
                placeholder='Blog structure and headings appear here...',
                elem_classes='outline-box'
            )

        # ── TAB 6: AGENT LOG ─────────────────────────────────
        with gr.TabItem('&#128196; Agent Log'):
            gr.HTML('<p style="color:#64748B;font-size:13px;padding:8px 4px">Real-time activity log from all 7 agents in the pipeline.</p>')
            log_out = gr.Textbox(
                label='LIVE AGENT ACTIVITY LOG',
                lines=22, interactive=False,
                placeholder='Agent-by-agent logs appear here after the pipeline runs...',
                elem_classes='log-box'
            )

    # ── FOOTER ──────────────────────────────────────────────
    gr.HTML(
        '<div style="display:flex;align-items:center;justify-content:space-between;'
        'background:#1A1D27;border:1px solid #2E3350;border-radius:10px;'
        'padding:12px 20px;margin-top:8px;">'
        '<span style="color:#475569;font-size:11px">'
        'Autonomous Agentic SEO Blog Writer &#8212; LangGraph + Groq + Serper &#8212; AI Credits: Claude &#183; Copilot &#183; Gemini &#183; ChatGPT &#183; Perplexity'
        '</span>'
        '<div style="display:flex;gap:12px">'
        '<a href="https://console.groq.com" target="_blank" style="color:#60A5FA;font-size:11px;font-weight:600;text-decoration:none">Groq</a>'
        '<a href="https://serper.dev" target="_blank" style="color:#60A5FA;font-size:11px;font-weight:600;text-decoration:none">Serper</a>'
        '<a href="https://langchain-ai.github.io/langgraph" target="_blank" style="color:#60A5FA;font-size:11px;font-weight:600;text-decoration:none">LangGraph</a>'
        '</div></div>'
    )

    # ── EVENTS ──────────────────────────────────────────────
    run_btn.click(
        fn=run_pipeline,
        inputs=[niche_dd, extra_in, model_dd],
        outputs=[blog_out, seo_out, research_out, social_out, log_out, outline_out, status_out]
    )
    clr_btn.click(
        fn=lambda: ('',)*7,
        outputs=[blog_out, seo_out, research_out, social_out, log_out, outline_out, status_out]
    )
    save_btn.click(fn=save_md, inputs=[blog_out], outputs=[save_file])


print('Dashboard ready! Launching...')
app.launch(share=True, debug=False)

Dashboard ready! Launching...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ada0a1a21b3d74639d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
